# S&P500/VIX Continuous Baseline

Continuous TC-VAE reference demo for the S&P500/VIX benchmark. The promoted method lives in the discrete notebook. This notebook does not require released checkpoints by default and does not run training unless `RUN_SMOKE=True`. Generated artefacts should stay under ignored `outputs/` paths.


## Setup

Load the package, locate the repository root, and display the selected continuous config.


In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
from pathlib import Path
from typing import Any

import pandas as pd
import yaml
from IPython.display import display

AUTO_SELECT_MODEL = True
MODEL_REGISTRY_PATH = "trained_models/model_registry.yaml"
MODEL_SELECTION_PROFILE = "balanced_market"
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_HEAVY = False

RUN_SMOKE = False
RUN_TRAINING = False
RUN_EVALUATION = False


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs" / "experiments"
        ).exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root()


def repo_path(path: str | Path) -> Path:
    candidate = Path(path).expanduser()
    if candidate.is_absolute():
        return candidate
    return (REPO_ROOT / candidate).resolve()


def display_path(path: str | Path) -> str:
    resolved = repo_path(path)
    try:
        return str(resolved.relative_to(REPO_ROOT))
    except ValueError:
        return str(resolved)


def load_yaml(path: str | Path) -> dict[str, Any]:
    resolved = repo_path(path)
    if not resolved.exists():
        return {}
    loaded = yaml.safe_load(resolved.read_text())
    return loaded if isinstance(loaded, dict) else {}


def load_json(path: str | Path) -> dict[str, Any] | None:
    resolved = repo_path(path)
    if not resolved.exists():
        return None
    return json.loads(resolved.read_text())


def print_command(command: list[str]) -> None:
    print(" ".join(shlex.quote(part) for part in command))


def maybe_run(command: list[str], *, enabled: bool, label: str) -> None:
    print(f"{label} command:")
    print_command(command)
    if enabled:
        subprocess.run(command, cwd=REPO_ROOT, check=True)
    else:
        print(f"{label} skipped; enable the matching RUN_* flag to execute it.")


print(f"Repository root: {REPO_ROOT}")
EXPERIMENT_ID = "sp500_vix"
CONFIG_PATH = "configs/experiments/sp500_vix_beta_cvae.yaml"
OUTPUT_DIR = "outputs/continuous/sp500_vix_beta_cvae"
BASE_DATA_DIR = "data/processed"
N_SAMPLE_TEST = 1000
print(f"Config: {display_path(CONFIG_PATH)}")

## Registered Model Selection


In [ ]:
from time_causal_vae.experiments.model_registry import load_registry, select_registered_model

REGISTERED_MODEL = None
CONTINUOUS_CHECKPOINT_CONVENTION = None
if AUTO_SELECT_MODEL:
    registry = load_registry(repo_path(MODEL_REGISTRY_PATH))
    REGISTERED_MODEL = select_registered_model(
        registry,
        experiment=EXPERIMENT_ID,
        family="continuous",
        profile=MODEL_SELECTION_PROFILE,
    )
    if REGISTERED_MODEL.config:
        CONFIG_PATH = REGISTERED_MODEL.config
    CONTINUOUS_CHECKPOINT_CONVENTION = REGISTERED_MODEL.checkpoint_paths.get(
        "checkpoint_convention"
    ) or REGISTERED_MODEL.checkpoint_paths.get("checkpoint_path")
    print(
        f"Registered continuous model: {REGISTERED_MODEL.candidate_id} "
        f"({REGISTERED_MODEL.selected_by})"
    )
    print(f"Config: {display_path(CONFIG_PATH)}")
    if CONTINUOUS_CHECKPOINT_CONVENTION:
        print(f"Checkpoint convention: {CONTINUOUS_CHECKPOINT_CONVENTION}")
    if not any(value is True for value in REGISTERED_MODEL.local_checkpoint_status.values()):
        print(
            "No local checkpoint was resolved by the registry. Supply a local final_model "
            "directory matching the convention before enabling RUN_EVALUATION."
        )
    if REGISTERED_MODEL.metrics:
        display(pd.DataFrame([REGISTERED_MODEL.metrics]))
    if REGISTERED_MODEL.missing_metrics:
        print("Missing metrics:", ", ".join(REGISTERED_MODEL.missing_metrics))
else:
    print("AUTO_SELECT_MODEL=False; using notebook-local config defaults.")

## Configuration Summary


In [ ]:
raw_config = load_yaml(CONFIG_PATH)
rows = []
for section, value in raw_config.items():
    if isinstance(value, dict):
        for key, item in value.items():
            if isinstance(item, (str, int, float, bool)) or item is None:
                rows.append({"section": section, "field": key, "value": item})
    elif isinstance(value, (str, int, float, bool)) or value is None:
        rows.append({"section": "root", "field": section, "value": value})
if rows:
    display(pd.DataFrame(rows))
else:
    print(f"Config missing or empty: {display_path(CONFIG_PATH)}")

## Dry-Run Commands

The smoke command builds data and model wiring without performing full training. The evaluation command is printed for use after a local continuous checkpoint exists.


In [ ]:
train_smoke_command = [
    "poetry",
    "run",
    "tcvae-train",
    "--config",
    display_path(CONFIG_PATH),
    "--output-dir",
    display_path(OUTPUT_DIR),
    "--epochs",
    "1",
    "--no-wandb",
    "--dry-run",
]
evaluation_command = [
    "poetry",
    "run",
    "tcvae-evaluate",
    "--config",
    display_path(CONFIG_PATH),
    "--model-dir",
    display_path(Path(OUTPUT_DIR) / "<training-run>" / "final_model"),
    "--output-dir",
    display_path(Path(OUTPUT_DIR) / "evaluation"),
    "--base-data-dir",
    display_path(BASE_DATA_DIR),
    "--n-sample-test",
    str(N_SAMPLE_TEST),
    "--seed",
    "99",
]
maybe_run(train_smoke_command, enabled=RUN_SMOKE, label="Continuous smoke")
print("\nEvaluation command for an existing checkpoint:")
print_command(evaluation_command)

## Expected Outputs

For a completed local run, keep generated summaries, figures, checkpoints, and executed notebooks below `outputs/`. The committed notebook should remain output-stripped.


## Torchview Diagram Helper

Torchview is optional and is not imported by package source. Load or instantiate a continuous model, assign it to the placeholder variable, and provide representative input data.

In [ ]:
try:
    from torchview import draw_graph
except ImportError:
    draw_graph = None


def show_model_graph(model: Any, *, input_data: Any | None = None, name: str = "model") -> None:
    if model is None:
        print(f"No {name} instance is loaded. Load or instantiate it first, then rerun this cell.")
        return
    if draw_graph is None:
        print(
            "Optional torchview support is unavailable. Install the notebooks group with `poetry install --with notebooks`."
        )
        return
    if input_data is None:
        print("Provide representative `input_data` for torchview, for example a small path batch.")
        return
    graph = draw_graph(model, input_data=input_data, expand_nested=True)
    display(graph.visual_graph)


CONTINUOUS_MODEL = None
show_model_graph(CONTINUOUS_MODEL, name="continuous TC-VAE")